# LeetCode 326 — Power of Three bằng Python

**Link đề:** https://leetcode.com/problems/power-of-three/
**Độ khó:** Easy · **Chủ đề:** Math, Recursion

```python
class Solution:
    def isPowerOfThree(self, n: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu định nghĩa **lũy thừa của 3**: `n == 3^x` với `x` là số nguyên **không âm**.
2. Thành thạo `%` và `//` để "bóc" thừa số 3 khỏi một số; biết bẫy **vòng lặp vô hạn** khi `n = 0`.
3. Viết 5 cách: chia lặp, đệ quy, đổi sang **hệ cơ số 3**, dùng **logarit** (và vì sao dễ sai), và mẹo **ước của 3^19** (không vòng lặp — follow-up của đề).
4. Trace từng bước, chứng minh tính đúng, so sánh Time/Space.
5. Tổng quát hóa cho "lũy thừa của k" và nhận ra khi nào mẹo "ước của lũy thừa lớn nhất" dùng được.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho số nguyên `n`. Trả về `True` nếu `n` là lũy thừa của 3, ngược lại `False`.

`n` là lũy thừa của 3 nếu tồn tại số nguyên `x` sao cho `n == 3^x`.

| Input `n` | Output | Giải thích |
|---:|:---:|---|
| `27` | `True` | `27 = 3^3` |
| `0` | `False` | không có `x` nào để `3^x = 0` |
| `-1` | `False` | `3^x` luôn dương |
| `1` | `True` | `1 = 3^0` (dễ quên!) |
| `9` | `True` | `3^2` |
| `45` | `False` | `45 = 9 · 5`, có thừa số 5 |

**Constraints:** `-2^31 <= n <= 2^31 - 1`.

**Follow up:** Bạn có giải được **không dùng vòng lặp / đệ quy** không?

Câu hỏi đặt ra:
- Với `x` âm, `3^x` là phân số (`3^-1 = 1/3`) → không phải số nguyên, nên chỉ cần xét `x >= 0`.
- Số âm và số 0 có bao giờ là lũy thừa của 3 không?

In [1]:
# Liệt kê các lũy thừa của 3 nằm trong giới hạn int 32-bit
powers = []
p = 1
while p <= 2**31 - 1:
    powers.append(p)
    p *= 3
print(powers)
print("Số lượng:", len(powers), "| lớn nhất:", powers[-1], "= 3 **", len(powers) - 1)

[1, 3, 9, 27, 81, 243, 729, 2187, 6561, 19683, 59049, 177147, 531441, 1594323, 4782969, 14348907, 43046721, 129140163, 387420489, 1162261467]
Số lượng: 20 | lớn nhất: 1162261467 = 3 ** 19


## 2. Nền tảng Python cho bài này

### 2.1. `**` (lũy thừa), `%` (chia lấy dư), `//` (chia lấy nguyên)
- `3 ** 4 = 81`.
- `n % 3 == 0` nghĩa là `n` chia hết cho 3.
- `n // 3` là thương nguyên — "bỏ đi một thừa số 3" khi `n` chia hết cho 3.
- `n / 3` cho **số thực (float)**, không dùng ở đây.

In [2]:
print(3 ** 4)        # 81
print(81 % 3, 82 % 3)   # 0 1
print(81 // 3)       # 27
print(81 / 3)        # 27.0 (float!)
print(type(81 // 3), type(81 / 3))

81
0 1
27
27.0
<class 'int'> <class 'float'>


### 2.2. Phân tích thừa số: một số là lũy thừa của 3 ⇔ chỉ có thừa số nguyên tố 3

Ví dụ `81 = 3·3·3·3` → chia liên tiếp cho 3: `81 → 27 → 9 → 3 → 1`. Kết thúc ở **1** → lũy thừa của 3.

`45 = 3·3·5` → `45 → 15 → 5`, dừng vì `5 % 3 != 0`, kết thúc ở **5 ≠ 1** → không phải.

In [3]:
for start in [81, 45]:
    n = start
    chain = [n]
    while n % 3 == 0:
        n //= 3
        chain.append(n)
    print(start, ":", " -> ".join(map(str, chain)), "| kết thúc ở", n)

81 : 81 -> 27 -> 9 -> 3 -> 1 | kết thúc ở 1
45 : 45 -> 15 -> 5 | kết thúc ở 5


### 2.3. Hệ cơ số 3 (ternary)

Trong hệ cơ số 3, lũy thừa của 3 có dạng `1`, `10`, `100`, `1000`… (giống `1, 10, 100` của hệ 10 với lũy thừa của 10).

Đổi sang cơ số 3: lặp lấy `n % 3` (chữ số cuối) và `n //= 3`.

In [4]:
def to_base3(n: int) -> str:
    if n == 0:
        return "0"
    digits = []
    while n > 0:
        digits.append(str(n % 3))
        n //= 3
    return "".join(reversed(digits))

for n in [1, 3, 9, 27, 45, 10]:
    print(n, "->", to_base3(n))

1 -> 1
3 -> 10
9 -> 100
27 -> 1000
45 -> 1200
10 -> 101


### 2.4. `math.log` và bẫy số thực

`math.log(n, 3)` cho `x` sao cho `3^x = n`, **nhưng là số thực** nên có sai số làm tròn.

In [5]:
import math
print(math.log(27, 3))           # mong đợi 3.0
print(math.log(243, 3))          # mong đợi 5.0 — thực tế là 4.999999999999999!
print(math.log(243, 3).is_integer())

3.0
4.999999999999999
False


## 3. Cách trực giác — chia cho 3 tới khi không chia được nữa

1. Nếu `n <= 0` → `False` (lũy thừa của 3 luôn dương).
2. Trong khi `n % 3 == 0`: `n //= 3`.
3. Cuối cùng `n == 1` → `True`, ngược lại `False`.

In [6]:
def is_power_of_three_loop(n: int) -> bool:
    if n <= 0:
        return False
    while n % 3 == 0:
        n //= 3
    return n == 1

for n in [27, 0, -1, 1, 9, 45, 2]:
    print(n, "->", is_power_of_three_loop(n))

27 -> True
0 -> False
-1 -> False
1 -> True
9 -> True
45 -> False
2 -> False


### 3.1. Cách đệ quy — cùng ý tưởng

```text
f(n) = True        nếu n == 1
f(n) = False       nếu n <= 0 hoặc n % 3 != 0
f(n) = f(n // 3)   ngược lại
```

In [7]:
def is_power_of_three_recursive(n: int) -> bool:
    if n == 1:
        return True
    if n <= 0 or n % 3 != 0:
        return False
    return is_power_of_three_recursive(n // 3)

print([n for n in range(-5, 100) if is_power_of_three_recursive(n)])

[1, 3, 9, 27, 81]


## 4. Nút thắt — follow-up: không vòng lặp, không đệ quy

Cách chia lặp tốn `O(log₃ n)` bước (tối đa ~19 bước với int 32-bit) — đã rất nhanh. Nhưng đề hỏi thêm: có cách **O(1)** không dùng vòng lặp không?

Ý tưởng thử đầu tiên là logarit: `x = log₃(n)`, kiểm tra `x` có nguyên không. Như cell 2.4 đã cho thấy, **sai số float** làm cách này sai với `243`. Ta cần một ý tưởng **thuần số nguyên**.

In [8]:
def is_power_of_three_log_naive(n: int) -> bool:
    if n <= 0:
        return False
    return math.log(n, 3).is_integer()

wrong = [p for p in powers if not is_power_of_three_log_naive(p)]
print("Các lũy thừa của 3 bị cách log ngây thơ trả SAI:", wrong)

Các lũy thừa của 3 bị cách log ngây thơ trả SAI: [243, 59049, 1594323, 14348907, 129140163]


## 5. Insight — mọi lũy thừa của 3 trong giới hạn đều là **ước** của `3^19`

- Lũy thừa của 3 lớn nhất ≤ `2^31 - 1` là `3^19 = 1162261467`.
- Vì 3 là **số nguyên tố**, các ước dương của `3^19` **chỉ** có dạng `3^k` với `0 <= k <= 19`.
- Vậy với `n > 0`: `n` là lũy thừa của 3 ⇔ `3^19 % n == 0`.

Chú ý: mẹo này chỉ đúng vì **3 là số nguyên tố**. Với "lũy thừa của 4", `4^15 % 2 == 0` nhưng 2 không phải lũy thừa của 4!

In [9]:
MAX_POW3 = 3 ** 19
print(MAX_POW3, MAX_POW3 <= 2**31 - 1, 3 ** 20 <= 2**31 - 1)
for n in [1, 27, 45, 81, 2, 1162261467]:
    print(n, "| 3^19 % n =", MAX_POW3 % n, "->", MAX_POW3 % n == 0)

1162261467 True False
1 | 3^19 % n = 0 -> True
27 | 3^19 % n = 0 -> True
45 | 3^19 % n = 27 -> False
81 | 3^19 % n = 0 -> True
2 | 3^19 % n = 1 -> False
1162261467 | 3^19 % n = 0 -> True


## 6. Lời giải chính (follow-up) — giảng từng đoạn rồi ghép

**Đoạn 1 — loại số không dương:** `n > 0`. Rất quan trọng: `3^19 % 0` gây `ZeroDivisionError`, và số âm như `-3` cũng chia hết `3^19` theo phép `%` của Python!

**Đoạn 2 — kiểm tra ước:** `1162261467 % n == 0`.

Ghép bằng `and` (ngắn mạch: nếu `n > 0` sai thì **không** tính vế sau, nên không chia cho 0).

In [10]:
print(MAX_POW3 % -3)          # 0 — số âm vẫn "chia hết", nên phải chặn n > 0
try:
    MAX_POW3 % 0
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

def is_power_of_three_divisor(n: int) -> bool:
    return n > 0 and 1162261467 % n == 0

print([n for n in range(-30, 100) if is_power_of_three_divisor(n)])

0
ZeroDivisionError: integer modulo by zero
[1, 3, 9, 27, 81]


### 6.1. Cách log "đúng cách" — làm tròn rồi kiểm tra lại bằng số nguyên

Nếu vẫn muốn dùng log: lấy `x = round(log(n, 3))`, rồi kiểm tra **chính xác** `3 ** x == n` bằng số nguyên.

In [11]:
def is_power_of_three_log_safe(n: int) -> bool:
    if n <= 0:
        return False
    x = round(math.log(n, 3))
    return 3 ** x == n

print(all(is_power_of_three_log_safe(p) for p in powers))
print([n for n in range(1, 1000) if is_power_of_three_log_safe(n)])

True
[1, 3, 9, 27, 81, 243, 729]


### 6.2. Cách hệ cơ số 3 — chuỗi phải là `"1"` theo sau toàn `"0"`

In [12]:
def is_power_of_three_base3(n: int) -> bool:
    if n <= 0:
        return False
    s = to_base3(n)
    return s[0] == "1" and s.count("0") == len(s) - 1

print([n for n in range(1, 1000) if is_power_of_three_base3(n)])

[1, 3, 9, 27, 81, 243, 729]


## 7. TRACE từng bước

### 7.1. Trace cách chia lặp

| Bước | n trước | n % 3 | Hành động | n sau |
|---:|---:|:---:|---|---:|
| 1 | 243 | 0 | chia 3 | 81 |
| 2 | 81 | 0 | chia 3 | 27 |
| 3 | 27 | 0 | chia 3 | 9 |
| 4 | 9 | 0 | chia 3 | 3 |
| 5 | 3 | 0 | chia 3 | 1 |
| 6 | 1 | 1 | **dừng** | 1 → `True` |

In [13]:
def trace_power_of_three(n: int) -> bool:
    print(f"Kiểm tra n = {n}")
    if n <= 0:
        print("  n <= 0 -> False ngay")
        return False
    step = 1
    while n % 3 == 0:
        print(f"  bước {step}: n = {n:>4}, n % 3 = 0 -> n //= 3 -> {n // 3}")
        n //= 3
        step += 1
    print(f"  dừng: n = {n}, n % 3 = {n % 3}; n == 1? {n == 1}")
    return n == 1

trace_power_of_three(243)
trace_power_of_three(162)     # 162 = 2 · 3^4
trace_power_of_three(1)
trace_power_of_three(0)

Kiểm tra n = 243
  bước 1: n =  243, n % 3 = 0 -> n //= 3 -> 81
  bước 2: n =   81, n % 3 = 0 -> n //= 3 -> 27
  bước 3: n =   27, n % 3 = 0 -> n //= 3 -> 9
  bước 4: n =    9, n % 3 = 0 -> n //= 3 -> 3
  bước 5: n =    3, n % 3 = 0 -> n //= 3 -> 1
  dừng: n = 1, n % 3 = 1; n == 1? True
Kiểm tra n = 162
  bước 1: n =  162, n % 3 = 0 -> n //= 3 -> 54
  bước 2: n =   54, n % 3 = 0 -> n //= 3 -> 18
  bước 3: n =   18, n % 3 = 0 -> n //= 3 -> 6
  bước 4: n =    6, n % 3 = 0 -> n //= 3 -> 2
  dừng: n = 2, n % 3 = 2; n == 1? False
Kiểm tra n = 1
  dừng: n = 1, n % 3 = 1; n == 1? True
Kiểm tra n = 0
  n <= 0 -> False ngay


False

### 7.2. Trace cách ước của `3^19`

Ta in phân tích của `n` để thấy vì sao `3^19 % n` bằng 0 hay không.

In [14]:
def trace_divisor(n: int) -> bool:
    if n <= 0:
        print(f"n = {n}: n > 0 sai -> False (không tính phép %)")
        return False
    r = 1162261467 % n
    print(f"n = {n}: 3^19 % n = {r} -> {r == 0}")
    return r == 0

for n in [243, 162, 1, 0, -27, 6]:
    trace_divisor(n)

n = 243: 3^19 % n = 0 -> True
n = 162: 3^19 % n = 81 -> False
n = 1: 3^19 % n = 0 -> True
n = 0: n > 0 sai -> False (không tính phép %)
n = -27: n > 0 sai -> False (không tính phép %)
n = 6: 3^19 % n = 3 -> False


## 8. Code nộp LeetCode

### Cách 1 — chia lặp (dễ hiểu, nên học trước)

In [15]:
class SolutionLoop:
    def isPowerOfThree(self, n: int) -> bool:
        if n <= 0:
            return False
        while n % 3 == 0:
            n //= 3
        return n == 1

print(SolutionLoop().isPowerOfThree(27), SolutionLoop().isPowerOfThree(0), SolutionLoop().isPowerOfThree(-1))

True False False


### Cách chính (follow-up) — ước của `3^19`, O(1)

In [16]:
class Solution:
    def isPowerOfThree(self, n: int) -> bool:
        return n > 0 and 1162261467 % n == 0

print(Solution().isPowerOfThree(27), Solution().isPowerOfThree(0), Solution().isPowerOfThree(-1))

True False False


### Cách 3 — tập hợp lũy thừa tính sẵn (cũng O(1) mỗi lần gọi)

**Set** là cấu trúc tra cứu `in` trung bình `O(1)`. Chỉ có 20 lũy thừa nên tạo sẵn rất rẻ.

In [17]:
class SolutionSet:
    POWERS = {3 ** k for k in range(20)}      # set comprehension: {0..19}

    def isPowerOfThree(self, n: int) -> bool:
        return n in self.POWERS

print(len(SolutionSet.POWERS), SolutionSet().isPowerOfThree(1162261467), SolutionSet().isPowerOfThree(6))

20 True False


## 9. Vì sao đúng?

**Cách chia lặp — invariant:** "`n` ban đầu là lũy thừa của 3 ⇔ `n` hiện tại là lũy thừa của 3".
- Khi `n % 3 == 0`, `n = 3·m`, và `n` là `3^x` (x ≥ 1) ⇔ `m` là `3^(x-1)`. Chia cho 3 giữ nguyên câu trả lời.
- Vòng lặp dừng khi `n % 3 != 0`. Lúc đó lũy thừa của 3 duy nhất không chia hết cho 3 là `3^0 = 1`. Nên trả `n == 1`.
- Vòng lặp **luôn dừng** khi `n > 0` vì `n` giảm thực sự mỗi bước (`n // 3 < n`). Với `n = 0`, `0 % 3 == 0` mãi mãi → phải chặn trước.

**Cách ước `3^19`:** 3 nguyên tố ⇒ ước dương của `3^19` đúng là `{3^0, …, 3^19}` (định lý phân tích thừa số nguyên tố duy nhất). Mọi lũy thừa của 3 trong miền int đều ≤ `3^19`, nên nằm trong tập này.

## 10. Time / Space Complexity và so sánh

Gọi `n` là giá trị đầu vào (không phải số chữ số).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Chia lặp | `O(log₃ n)` (≤ 19 vòng) | `O(1)` | dễ hiểu, an toàn |
| Đệ quy | `O(log₃ n)` | `O(log₃ n)` stack | giống chia lặp |
| Hệ cơ số 3 | `O(log₃ n)` | `O(log₃ n)` chuỗi | minh họa đẹp |
| Log ngây thơ | `O(1)` | `O(1)` | **SAI** do float |
| Log + kiểm tra lại | `O(1)`* | `O(1)` | đúng; *coi `3 ** x` là hằng vì x ≤ 19 |
| **Ước của 3^19** | **`O(1)`** | **`O(1)`** | follow-up, chỉ đúng vì 3 nguyên tố |
| Set tính sẵn | `O(1)` trung bình | `O(20)` | cũng không vòng lặp lúc truy vấn |

## 11. Bộ test — mẫu đề, biên, đối chiếu

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề: 27", sol.isPowerOfThree(27), True)
check("đề: 0", sol.isPowerOfThree(0), False)
check("đề: -1", sol.isPowerOfThree(-1), False)
check("1 = 3^0", sol.isPowerOfThree(1), True)
check("3", sol.isPowerOfThree(3), True)
check("45 = 9·5", sol.isPowerOfThree(45), False)
check("số âm -27", sol.isPowerOfThree(-27), False)
check("3^19", sol.isPowerOfThree(1162261467), True)
check("3^19 - 1", sol.isPowerOfThree(1162261466), False)
check("max int", sol.isPowerOfThree(2**31 - 1), False)
check("min int", sol.isPowerOfThree(-2**31), False)
check("2 (không chia hết 3)", sol.isPowerOfThree(2), False)

PASS đề: 27: True
PASS đề: 0: False
PASS đề: -1: False
PASS 1 = 3^0: True
PASS 3: True
PASS 45 = 9·5: False
PASS số âm -27: False
PASS 3^19: True
PASS 3^19 - 1: False
PASS max int: False
PASS min int: False
PASS 2 (không chia hết 3): False


In [19]:
import random
random.seed(326)
truth = set(powers)
methods = [is_power_of_three_loop, is_power_of_three_recursive, is_power_of_three_divisor,
           is_power_of_three_log_safe, is_power_of_three_base3,
           SolutionLoop().isPowerOfThree, Solution().isPowerOfThree, SolutionSet().isPowerOfThree]
samples = list(range(-50, 2000)) + powers + [p - 1 for p in powers] + [p + 1 for p in powers] + [2 * p for p in powers]
samples += [random.randint(-2**31, 2**31 - 1) for _ in range(3000)]
for n in samples:
    expected = n in truth
    for f in methods:
        assert f(n) == expected, (f, n)
print(f"{len(samples)} giá trị × {len(methods)} cách: tất cả khớp với tập lũy thừa chuẩn")

5130 giá trị × 8 cách: tất cả khớp với tập lũy thừa chuẩn


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên chặn `n <= 0` → vòng lặp vô hạn với `n = 0`
`0 % 3 == 0` và `0 // 3 == 0` → `while` chạy mãi. Ta demo bằng cách giới hạn số vòng.

In [20]:
def loop_without_guard(n: int, max_steps: int = 10) -> str:
    steps = 0
    while n % 3 == 0:           # SAI: không chặn n <= 0
        n //= 3
        steps += 1
        if steps >= max_steps:
            return f"vẫn chạy sau {steps} vòng, n = {n} -> vòng lặp vô hạn!"
    return f"dừng với n = {n}"

print(loop_without_guard(0))
print(loop_without_guard(27))

vẫn chạy sau 10 vòng, n = 0 -> vòng lặp vô hạn!
dừng với n = 1


### Lỗi 2: Dùng `/` thay vì `//`
`n / 3` tạo float; với số lớn có thể mất chính xác và `n == 1` so sánh float.

### Lỗi 3: Quên `1 = 3^0`
Viết `return n == 3` hoặc bắt đầu kiểm tra từ `n >= 3` → `isPowerOfThree(1)` sai.

### Lỗi 4: Tin vào `math.log(n, 3).is_integer()`
Sai với `243` như đã thấy.

In [21]:
def wrong_start_from_three(n: int) -> bool:
    if n < 3:                   # SAI: bỏ mất n = 1
        return False
    while n % 3 == 0:
        n //= 3
    return n == 1

print("1 ->", wrong_start_from_three(1), "(đúng phải là True)")
print("log ngây thơ với 243 ->", is_power_of_three_log_naive(243), "(đúng phải là True)")
big = 3 ** 40 + 3                 # số rất lớn (vượt 2^53, float không biểu diễn chính xác)
print("float `/`:", int(big / 3) == big // 3, "| int `//`:", (big // 3) * 3 == big)   # / làm mất chính xác

1 -> False (đúng phải là True)
log ngây thơ với 243 -> False (đúng phải là True)
float `/`: False | int `//`: True


### Lỗi 5: Áp dụng mẹo "ước của lũy thừa lớn nhất" cho cơ số **không nguyên tố**
`4^15 % 2 == 0` nhưng 2 không phải lũy thừa của 4 → mẹo sai với cơ số 4, 6, 8, 9, 10…

In [22]:
MAX_POW4 = 4 ** 15
print("4^15 % 2 =", MAX_POW4 % 2, "-> mẹo nói 2 là lũy thừa của 4: SAI")
print("4^15 % 8 =", MAX_POW4 % 8, "-> mẹo nói 8 là lũy thừa của 4: SAI")

4^15 % 2 = 0 -> mẹo nói 2 là lũy thừa của 4: SAI
4^15 % 8 = 0 -> mẹo nói 8 là lũy thừa của 4: SAI


## 13. Mẹo nhận dạng pattern

- **"Có phải lũy thừa của k?"** → mặc định: chặn `n <= 0`, chia lặp cho `k`, kiểm tra kết thúc ở 1.
- **Follow-up không vòng lặp** + `k` nguyên tố + miền giá trị bị chặn → `n > 0 and MAX_POW_K % n == 0`.
- `k = 2` → mẹo bit `n & (n - 1) == 0` (bài 231). `k = 4` → mẹo bit + vị trí bit (bài 342).
- Thấy `math.log` cho bài số nguyên → luôn nghi ngờ sai số; kiểm tra lại bằng số nguyên.
- Thấy `while n % k == 0` → nhớ trường hợp `n = 0`.

## 14. Biến thể

### Biến thể 1 — lũy thừa của `k` bất kỳ (k ≥ 2)

In [23]:
def is_power_of_k(n: int, k: int) -> bool:
    if n <= 0:
        return False
    while n % k == 0:
        n //= k
    return n == 1

print([n for n in range(1, 200) if is_power_of_k(n, 5)])
print([n for n in range(1, 200) if is_power_of_k(n, 6)])

[1, 5, 25, 125]
[1, 6, 36]


### Biến thể 2 — trả về số mũ `x` (hoặc -1)
### Biến thể 3 — Check if Number is a Sum of Powers of Three (LeetCode 1780)
`n` viết được thành tổng các lũy thừa **phân biệt** của 3 ⇔ biểu diễn cơ số 3 của `n` chỉ gồm chữ số 0 và 1.

In [24]:
def log3_exact(n: int) -> int:
    if n <= 0:
        return -1
    x = 0
    while n % 3 == 0:
        n //= 3
        x += 1
    return x if n == 1 else -1

def is_sum_of_distinct_powers_of_three(n: int) -> bool:
    while n > 0:
        if n % 3 == 2:
            return False
        n //= 3
    return True

print(log3_exact(81), log3_exact(82), log3_exact(1))           # 4 -1 0
print(is_sum_of_distinct_powers_of_three(12), to_base3(12))     # True  110 (9 + 3)
print(is_sum_of_distinct_powers_of_three(21), to_base3(21))     # False 210

4 -1 0
True 110
False 210


## 15. Bài tập tự luyện (TODO — không chạy sẵn)

### Bài 1 — Tính tay
Trace cách chia lặp cho `n = 54` và `n = 729`. Mỗi số dừng ở đâu? Kết quả?

### Bài 2 — Dự đoán output
Đoán: `1162261467 % 243`, `1162261467 % 486`, `to_base3(13)`.

### Bài 3 — Điền chỗ trống

In [25]:
def exercise_power_of_three(n: int) -> bool:
    # TODO 1: chặn n <= 0
    # TODO 2: while n % 3 == 0: n //= 3
    # TODO 3: return n == 1
    pass

# assert exercise_power_of_three(27) is True
# assert exercise_power_of_three(0) is False
# assert exercise_power_of_three(1) is True

### Bài 4 — Tự viết: lũy thừa của 5 không vòng lặp
5 là số nguyên tố. Hãy tìm lũy thừa của 5 lớn nhất ≤ `2^31 - 1` (bằng code), rồi viết hàm O(1).

In [26]:
def exercise_power_of_five(n: int) -> bool:
    # TODO
    pass

# assert exercise_power_of_five(125) is True
# assert exercise_power_of_five(0) is False
# assert exercise_power_of_five(50) is False

### Bài 5 — Biến thể: lũy thừa của 3 nhỏ nhất ≥ n
Viết `exercise_next_power_of_three(n)` với `n >= 1`.

In [27]:
def exercise_next_power_of_three(n: int) -> int:
    # TODO
    pass

# assert exercise_next_power_of_three(1) == 1
# assert exercise_next_power_of_three(10) == 27
# assert exercise_next_power_of_three(27) == 27

### Bài 6 — Mini challenge: tổng các lũy thừa phân biệt của 3
Viết `exercise_sum_of_powers(n)` trả về **danh sách các lũy thừa của 3 phân biệt** có tổng bằng `n` (tăng dần), hoặc `[]` nếu không được. Ví dụ `12 -> [3, 9]`, `91 -> [1, 9, 81]`, `21 -> []`.

In [28]:
def exercise_sum_of_powers(n: int) -> list[int]:
    # TODO: duyệt chữ số cơ số 3; chữ số 1 ở vị trí k -> thêm 3**k; chữ số 2 -> trả []
    pass

# assert exercise_sum_of_powers(12) == [3, 9]
# assert exercise_sum_of_powers(91) == [1, 9, 81]
# assert exercise_sum_of_powers(21) == []

## 16. Gợi ý 3 mức

**Bài 1** — (1) Mỗi bước kiểm tra `% 3`. (2) `54 = 2 · 27`. (3) `729 = 3^6`.

**Bài 2** — (1) `243 = 3^5`. (2) `486 = 2 · 3^5`, có thừa số 2. (3) `13 = 9 + 3 + 1`.

**Bài 3** — (1) Thứ tự: chặn, lặp, so sánh. (2) Dùng `//=`. (3) Xem lại cell 3.

**Bài 4** — (1) Dùng vòng `while p * 5 <= 2**31 - 1`. (2) Kết quả là `5^13 = 1220703125`. (3) `n > 0 and 1220703125 % n == 0`.

**Bài 5** — (1) Bắt đầu `p = 1`. (2) Nhân 3 khi `p < n`. (3) Trả `p`.

**Bài 6** — (1) Giống `to_base3`. (2) Giữ biến `power` nhân 3 sau mỗi chữ số. (3) Chữ số 2 → `return []`.

## 17. Đáp án tham khảo

**Bài 1:** `54 → 18 → 6 → 2` dừng ở 2 → `False`. `729 → 243 → 81 → 27 → 9 → 3 → 1` → `True`.

**Bài 2:** `0`, `243` (khác 0), `"111"`.

In [29]:
def answer_power_of_three(n: int) -> bool:
    if n <= 0:
        return False
    while n % 3 == 0:
        n //= 3
    return n == 1

p5 = 1
while p5 * 5 <= 2**31 - 1:
    p5 *= 5
print("Lũy thừa của 5 lớn nhất:", p5)

def answer_power_of_five(n: int) -> bool:
    return n > 0 and 1220703125 % n == 0

def answer_next_power_of_three(n: int) -> int:
    p = 1
    while p < n:
        p *= 3
    return p

def answer_sum_of_powers(n: int) -> list[int]:
    result = []
    power = 1
    while n > 0:
        digit = n % 3
        if digit == 2:
            return []
        if digit == 1:
            result.append(power)
        n //= 3
        power *= 3
    return result

assert p5 == 1220703125
assert not answer_power_of_three(54) and answer_power_of_three(729)
assert 1162261467 % 243 == 0 and 1162261467 % 486 == 243 and to_base3(13) == "111"
assert answer_power_of_three(27) is True and answer_power_of_three(0) is False and answer_power_of_three(1) is True
assert answer_power_of_five(125) is True and answer_power_of_five(0) is False and answer_power_of_five(50) is False
assert all(answer_power_of_five(n) == is_power_of_k(n, 5) for n in range(-10, 5000))
assert answer_next_power_of_three(1) == 1 and answer_next_power_of_three(10) == 27 and answer_next_power_of_three(27) == 27
assert answer_sum_of_powers(12) == [3, 9] and answer_sum_of_powers(91) == [1, 9, 81] and answer_sum_of_powers(21) == []
print("Đáp án tham khảo PASS toàn bộ test!")

Lũy thừa của 5 lớn nhất: 1220703125
Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def isPowerOfThree(self, n: int) -> bool:
        # Cách dễ hiểu
        # if n <= 0: return False
        # while n % 3 == 0: n //= 3
        # return n == 1
        return n > 0 and 1162261467 % n == 0     # 3^19, follow-up O(1)
```

**Template "lũy thừa của k":** chặn `n <= 0` → chia lặp cho `k` → kiểm tra `== 1`. Nếu `k` nguyên tố và miền bị chặn: `n > 0 and K_MAX % n == 0`.

**Checklist trước khi submit:**
- [ ] Đã chặn `n <= 0` (tránh vòng lặp vô hạn / chia cho 0 / số âm)?
- [ ] `1` trả về `True`?
- [ ] Dùng `//` chứ không phải `/`?
- [ ] Không tin trực tiếp kết quả `math.log`?
- [ ] Hằng số đúng `1162261467 = 3^19`?